# UnCLIP: the model behind Dall-E-2 image generation model of Open AI



In this tutorial, we will create and train an UnCLIP model with [TorchDiff](https://loqmansamani.github.io/torchdiff/) diffusion library, a diffusion library built on top of the [PyTorch](https://pytorch.org/) API, which consist of the unclip model (Dall-E-2) implemented based on it main paper [Hierarchical Text-Conditional Image Generation with CLIP Latents](https://arxiv.org/abs/2204.06125). Note that this tutorial will not train a model in real-time; instead, it will demonstrate how to train an UnCLIP model using the TorchDiff API.

## Table of Contents

- [Data Preparation](#data-preparation)
- [Train a  diffusion prior model](#train-diffusion-prior)
- [Train the decoder model](#train-decoder)
- [Train the first upsampler: 64x64 -> 256x256](#train-first-upsampler)
- [Train the second upsampler: 256x256 -> 1024x1024](#train-second-upsampler)
- [Sampling pipeline](#sampling-pipeline)

In [1]:
# Import all necessary libraries and modules for training an UnCLIP model
import os
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import torch
import torch.nn as nn
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader, Subset




# Import required classes from the UnCLIP module of TorchDiff

# main diffusion classes needed to apply diffusion
from unclip import VarianceSchedulerUnCLIP, ForwardUnCLIP, ReverseUnCLIP

# classes related to clip models used for embedding images and prompts (texts) into clip latent space
from unclip import CLIPEncoder, CLIPContextProjection, CLIPEmbeddingProjection

# classes related to prior transformer based model of unclip
from unclip import UnCLIPTransformerPrior, TrainUnCLIPPrior

# classes related to decoder model of unclip
from unclip import UnClipDecoder, TrainUnClipDecoder

# classes related to unsampler models
from unclip import UpsamplerUnCLIP, TrainUpsamplerUnCLIP

# the sampler pipe line of unclip
from unclip import SampleUnCLIP

# Import utility functions from the TorchDiff utils module
from utils import NoisePredictor, TextEncoder, Metrics

## Data Preparation

For this tutorial, we will use the [CIFAR-10](https://docs.pytorch.org/vision/main/generated/torchvision.datasets.CIFAR10.html) dataset from [Torchvision](https://pytorch.org/vision/stable/) with a descriptive caption (mock caption which we will add to each image only for education purposes in this toturial). This dataset consists of low-resolution RBG images, making it suitable for this tutorial.

In [2]:
#Use CIFAR-10 with descriptive captions
class CIFAR10WithCaptions(Dataset):
    def __init__(self, cifar_dataset):
        self.dataset = cifar_dataset
        self.class_names = [
            'airplane', 'automobile', 'bird', 'cat', 'deer',
            'dog', 'frog', 'horse', 'ship', 'truck'
        ]
        # More descriptive templates
        self.templates = [
            "A photo of a {}",
            "An image of a {}",
            "A picture of a {}",
            "This is a {}",
        ]

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        class_name = self.class_names[label]
        # Use different templates for variety
        template = self.templates[idx % len(self.templates)]
        caption = template.format(class_name)
        return image, caption


# Updated transforms for CLIP
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load CIFAR-10 with captions
cifar_train = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
cifar_test = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 170M/170M [00:26<00:00, 6.53MB/s]


### Using a Subset of the CIFAR10 Dataset

This tutorial will use a small subset of the CIFAR10 dataset for training and validation. Specifically.

In [3]:
train_dataset = CIFAR10WithCaptions(cifar_train)
test_dataset = CIFAR10WithCaptions(cifar_test)

# Small subset for testing
train_subset_indices = torch.randperm(len(train_dataset))[:100]
test_subset_indices = torch.randperm(len(test_dataset))[:20]

train_subset = Subset(train_dataset, train_subset_indices)
test_subset = Subset(test_dataset, test_subset_indices)

# DataLoaders
t_loader = DataLoader(train_subset, batch_size=32, shuffle=True, pin_memory=True)
val = DataLoader(test_subset, batch_size=10, shuffle=False, pin_memory=True)